<h1>Data preprocessing

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             precision_recall_curve, roc_curve, auc,
                             roc_auc_score, average_precision_score,
                             f1_score, recall_score)
from imblearn.over_sampling import SMOTE
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import lightgbm as lgb

plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = os.path.abspath(os.environ.get("AMD_PAMD_DATA_DIR", os.path.join(os.getcwd(), "data")))
OUTPUT_DIR = os.path.abspath(os.environ.get("AMD_PAMD_OUTPUT_DIR", os.path.join(os.getcwd(), "outputs")))
os.makedirs(OUTPUT_DIR, exist_ok=True)
HEAD_DIR = os.path.join(BASE_DIR, "Sequential structure of the heat map", "head")
TAIL_DIR = os.path.join(BASE_DIR, "Sequential structure of the heat map", "tail")
DATASET_PATH = os.path.join(BASE_DIR, "Dataset.xlsx")


def natural_key(s):
    """Natural sorting key to keep files in sequential order, such as A1, A2, ..., A10."""
    return [int(x) if x.isdigit() else x for x in re.split(r'(\d+)', s)]


def read_sdf_folder_custom(folder_path):
    """Reads all SDF files in a folder and returns a list of RDKit molecule objects."""
    molecules = []
    for filename in sorted(os.listdir(folder_path), key=natural_key):
        if filename.endswith('.sdf'):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                molblock = f.read()
            mol = Chem.MolFromMolBlock(molblock)
            if mol is not None:
                molecules.append(mol)
    return molecules


all_desc_names = [desc[0] for desc in Descriptors._descList]


def calc_208_descriptors(mol):
    """Calculates 208 RDKit descriptors for a molecule."""
    desc_values = []
    for name in all_desc_names:
        func = getattr(Descriptors, name)
        try:
            val = func(mol)
        except:
            val = 0.0  # Assign 0 if an error occurs
        desc_values.append(val)
    return desc_values


def calc_features(mol):
    """Calculates combined features (Morgan fingerprint + 208 descriptors) for a molecule."""
    # 2048-bit Morgan fingerprint
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    fp_arr = list(fp)

    # 208 descriptors
    desc208 = calc_208_descriptors(mol)

    # Concatenate fingerprint and descriptors
    features = fp_arr + desc208
    return features


def feat_name(idx):
    """Returns the name of the feature given its index."""
    if idx < 2048:
        return f"Morgan_{idx}"
    else:
        desc_idx = idx - 2048
        return all_desc_names[desc_idx]


# Paths point to the current workspace data folder
head_mols = read_sdf_folder_custom(HEAD_DIR)
tail_mols = read_sdf_folder_custom(TAIL_DIR)

print(f"The number of head molecules in the training set: {len(head_mols)}")
print(f"The number of tail molecules in the training set: {len(tail_mols)}")

head_features = [calc_features(m) for m in head_mols]
tail_features = [calc_features(m) for m in tail_mols]

head_index = [f"A{i+1}" for i in range(len(head_features))]
tail_index = [f"O{i+1}" for i in range(len(tail_features))]

head_df = pd.DataFrame(head_features, index=head_index)
tail_df = pd.DataFrame(tail_features, index=tail_index)

df = pd.read_excel(DATASET_PATH, index_col=0)  # Rows: head molecule index, Columns: tail molecule index

X = []
y = []

for h in df.index:
    for t in df.columns:
        if str(h) in head_df.index and str(t) in tail_df.index:
            feat = list(head_df.loc[str(h)]) + list(tail_df.loc[str(t)])
            X.append(feat)
            val = df.loc[h, t]
            label = 1 if val >= 2000 else 0  # 1 for good (>= 2000), 0 for poor
            y.append(label)

X = pd.DataFrame(X)
y = pd.Series(y)

print(f"Training sample size: {len(y)}")
print(f"The ratio of positive to negative samples:\n{y.value_counts()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#################################################################
#          Unified feature preprocessing for all four models
#################################################################
# Low-variance feature removal and standardization are fitted only on the training set.
# The same processed feature matrix is then used for RF, XGBoost, LightGBM, and LR.

variance_selector = VarianceThreshold(threshold=0.0)
X_train_var = variance_selector.fit_transform(X_train)
X_test_var = variance_selector.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_var)
X_test_scaled = scaler.transform(X_test_var)

print(f"Training set shape after low-variance feature removal: {X_train_scaled.shape}")
print(f"Test set shape after low-variance feature removal: {X_test_scaled.shape}")

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Training set shape after SMOTE: {X_train_res.shape}")
print(f"Target distribution after SMOTE:\n{y_train_res.value_counts()}")

# Save preprocessing objects for reproducibility
joblib.dump(variance_selector, os.path.join(OUTPUT_DIR, 'variance_selector.pkl'))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'standard_scaler.pkl'))

# Results will be collected and printed at the end
all_results = []


def evaluate_model(model_name, y_prob, threshold=0.5):
    """Evaluates model performance using a unified threshold strategy."""
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    neg_recall = recall_score(y_test, y_pred, pos_label=0, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    print(f"Threshold ({model_name}): {threshold:.4f}")
    print(f"Accuracy rate ({model_name}): {acc:.4f}")
    print(f"F1 score ({model_name}): {f1:.4f}")
    print(f"Negative-class recall ({model_name}): {neg_recall:.4f}")
    print(f"Macro F1 ({model_name}): {macro_f1:.4f}")
    print("Classification report:")
    report_df = pd.DataFrame(
        classification_report(y_test, y_pred, digits=4, zero_division=0, output_dict=True)
    ).T
    report_df = report_df.drop(columns=["support"], errors="ignore")
    print(report_df.round(4))
    print(f"ROC AUC ({model_name}): {roc_auc:.4f}")
    print(f"PR AUC ({model_name}): {pr_auc:.4f}")

    all_results.append({
        'Model': model_name,
        'Accuracy': acc,
        'F1': f1,
        'ROC AUC': roc_auc,
        'PR AUC': pr_auc,
        'Negative-class recall': neg_recall,
        'macro F1': macro_f1
    })

    return y_pred


<h1>Train the comparison model

In [ ]:
#################################################################
#                           Model 1: Random Forest
#################################################################
print("\n" + "="*20 + " Model 1: Random Forest " + "="*20 + "\n")

param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [10, 20, 30, 40, None],
    'min_samples_split': [2, 5, 7, 9],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train_res, y_train_res)
best_rf_model = grid.best_estimator_
print(f"Optimal model parameters (Random Forest): {grid.best_params_}")

probs_rf = best_rf_model.predict_proba(X_test_scaled)[:, 1]
y_pred_rf = evaluate_model('Random Forest', probs_rf, threshold=0.5)

joblib.dump(best_rf_model, os.path.join(OUTPUT_DIR, 'best_random_forest_model.pkl'))


In [ ]:
#################################################################
#                Model 2: XGBoost
#################################################################
print("\n" + "="*20 + " Model 2: XGBoost " + "="*20 + "\n")

param_dist_xgb = {
    'n_estimators': [200, 400, 600],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0.1, 0.5, 1],
    'min_child_weight': [1, 5, 10],
    'scale_pos_weight': [1.0]
}

xgb = XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

random_search_xgb.fit(X_train_res, y_train_res)
best_xgb_model = random_search_xgb.best_estimator_
print(f"Optimal parameters (XGBoost): {random_search_xgb.best_params_}")

y_prob_xgb = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = evaluate_model('XGBoost', y_prob_xgb, threshold=0.5)

joblib.dump(best_xgb_model, os.path.join(OUTPUT_DIR, 'best_xgboost_model.pkl'))


In [ ]:
#################################################################
#                   Model 3: LightGBM (LGBM)
#################################################################
print("\n" + "="*20 + " Model 3: LightGBM (LGBM) " + "="*20 + "\n")

param_dist_lgbm = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [4, 6, 8, 10, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 40, 50],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'scale_pos_weight': [1.0]
}

lgbm = lgb.LGBMClassifier(objective='binary', random_state=42, verbosity=-1)

random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist_lgbm,
    n_iter=50,
    scoring='f1',
    cv=3,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

random_search_lgbm.fit(X_train_res, y_train_res)
print("Optimal parameters (LightGBM):", random_search_lgbm.best_params_)
best_lgbm_model = random_search_lgbm.best_estimator_

y_prob_lgbm = best_lgbm_model.predict_proba(X_test_scaled)[:, 1]
y_pred_lgbm = evaluate_model('LightGBM', y_prob_lgbm, threshold=0.5)

joblib.dump(best_lgbm_model, os.path.join(OUTPUT_DIR, 'best_lightgbm_model.pkl'))


In [ ]:
#################################################################
#                      Model 4: Logistic Regression
#################################################################
print("\n" + "="*20 + " Model 4: Logistic Regression " + "="*20 + "\n")

from sklearn.linear_model import LogisticRegression

param_grid_enet = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'l1_ratio': [0.1, 0.5, 0.9],
    'class_weight': ['balanced', None]
}

lr_enet = LogisticRegression(max_iter=3000, random_state=42)

grid_enet = GridSearchCV(
    lr_enet,
    param_grid_enet,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

grid_enet.fit(X_train_res, y_train_res)
print(f"Optimal parameters ElasticNet: {grid_enet.best_params_}")

best_enet = grid_enet.best_estimator_

probs_enet = best_enet.predict_proba(X_test_scaled)[:, 1]
y_pred_enet = evaluate_model('Logistic Regression', probs_enet, threshold=0.5)

joblib.dump(best_enet, os.path.join(OUTPUT_DIR, 'best_logistic_regression_model.pkl'))
